<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Подготовка" data-toc-modified-id="Подготовка-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Подготовка</a></span><ul class="toc-item"><li><span><a href="#Загрузка-данных" data-toc-modified-id="Загрузка-данных-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Загрузка данных</a></span></li><li><span><a href="#Предобработка." data-toc-modified-id="Предобработка.-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Предобработка.</a></span></li><li><span><a href="#Подготовка." data-toc-modified-id="Подготовка.-1.3"><span class="toc-item-num">1.3&nbsp;&nbsp;</span>Подготовка.</a></span></li></ul></li><li><span><a href="#Обучение" data-toc-modified-id="Обучение-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Обучение</a></span></li><li><span><a href="#Выводы" data-toc-modified-id="Выводы-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Выводы</a></span></li><li><span><a href="#Чек-лист-проверки" data-toc-modified-id="Чек-лист-проверки-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Чек-лист проверки</a></span></li></ul></div>

# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

Постройте модель со значением метрики качества *F1* не меньше 0.75. 

**Инструкция по выполнению проекта**

1. Загрузите и подготовьте данные.
2. Обучите разные модели. 
3. Сделайте выводы.

Для выполнения проекта применять *BERT* необязательно, но вы можете попробовать.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.

In [1]:
!pip install nltk -q
   

In [2]:
!pip install lightgbm -q

In [3]:
import numpy as np

import pandas as pd
from tqdm import tqdm
import scipy.stats as st
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import timeit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier
nltk.download('punkt')
nltk.download('punkt_tab')           
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...

True

In [4]:
RANDOM_STATE = 52
TEST_SIZE = 0.25
stops = set(stopwords.words('english'))

## Подготовка

### Загрузка данных

In [5]:
data = pd.read_csv("toxic_comments.csv",index_col=0)
data.head()

,text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 159292 entries, 0 to 159450
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    159292 non-null  object
 1   toxic   159292 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.6+ MB


В данных 159292 твитов.

### Предобработка.

In [7]:
data.duplicated().sum()

0

В данных нет  дубликатов.

### Подготовка.

## Обучение

In [8]:
# Функция для приведения POS-тегов в формат WordNet
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)  # По умолчанию - существительное

# Функция для предобработки текста
def preprocess_text(text):
    text = text.lower()  # Приведение к нижнему регистру
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Удаление лишних символов
    tokens = word_tokenize(text)  # Токенизация
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in tokens]
    return ' '.join(lemmatized_tokens)

In [9]:
%%time
tqdm.pandas()
data['text'] = data['text'].progress_apply(preprocess_text)
data

100%|█████████████████████████████████████████████████████████████████████████| 159292/159292 [20:13<00:00, 131.31it/s]

CPU times: total: 19min 13s
Wall time: 20min 13s


,text,toxic
0,explanation why the edits make under my userna...,0
1,daww he match this background colour im seemin...,0
2,hey man im really not try to edit war it just ...,0
3,more i cant make any real suggestion on improv...,0
4,you sir be my hero any chance you remember wha...,0
...,...,...
159446,and for the second time of ask when your view ...,0
159447,you should be ashamed of yourself that be a ho...,0
159448,spitzer umm there no actual article for prosti...,0
159449,and it look like it be actually you who put on...,0


In [10]:
X = data['text'] 
y = data['toxic']




X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [16]:


tfidf = TfidfVectorizer(max_features=5000, lowercase=False, sublinear_tf=False)
X_train_tfidf = tfidf.fit_transform(X_train)  
X_test_tfidf = tfidf.transform(X_test)

X_search, _, y_search, _ = train_test_split(
    X_train_tfidf, y_train, train_size=30000, random_state=42, stratify=y_train
)

pipe = LGBMClassifier(objective="binary", random_state=42, n_jobs=-1, verbose=-1)

param_dist = {
    'max_depth': [15, 20],
    'learning_rate': [0.1, 0.3],
    'n_estimators': [100, 200],
    'num_leaves': [50],
    'min_data_in_leaf': [20],
}

random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=4,
    cv=2,
    n_jobs=-1,
    verbose=2,
    scoring='f1',
    random_state=42,
    error_score=np.nan
)




In [17]:
random_search.fit(X_search, y_search)
best_params = random_search.best_params_
final_model = LGBMClassifier(**best_params, objective="binary", random_state=42, n_jobs=-1)
final_model.fit(X_train_tfidf, y_train)

y_pred = final_model.predict(X_test_tfidf)
f1 = f1_score(y_test, y_pred)
print(f"F1: {f1:.4f}")

Fitting 2 folds for each of 4 candidates, totalling 8 fits
F1: 0.7783


In [18]:
print('Метрика f1 =',f1)

Метрика f1 = 0.778318276580959


Был использован перебор параметров с помощью RandomizedSearchCV для модели LGBMClassifier, итоговая модель показывает метрика f1 на тестовой выборке в 0.778

## Выводы

Было проведено первичное ознакомление с данными. В таблице находиться информация о 159292 твитах.

При помощи пайплайна и перебора гиперпараметров была обучена модель LGBMClassifier, с итоговой метрикой f1 = 0.778, что подходит под условия задачи.

Для улучшения результат требуется собрать данные о большем количестве твитов разного показателя токсичности, а также более мощное оборудование, так как домашний компьютер уже требдует много времени для выполнения подобных задач.